In [ ]:
!pip install -q openai-whisper language-tool-python
!apt-get update && apt-get install -y ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.5/800.5 kB 17.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.7/54.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.3/54.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.6 MB/s e

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
zip_path = "/content/drive/MyDrive/shl-intern-hiring-assessment.zip"


In [ ]:
import zipfile
import os

unzip_dir = "/content/data/dataset"
os.makedirs(unzip_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(unzip_dir)

print("✅ Dataset extracted to", unzip_dir)


✅ Dataset extracted to /content/data/dataset


In [ ]:
import os

# Recursively print directory structure
for root, dirs, files in os.walk("/content"):
    for name in files:
        if name.endswith(".csv") or name.endswith(".wav"):
            print(os.path.join(root, name))


/content/data/dataset/dataset/test.csv
/content/data/dataset/dataset/train.csv
/content/data/dataset/dataset/sample_submission.csv
/content/data/dataset/dataset/audios_train/audio_1147.wav
/content/data/dataset/dataset/audios_train/audio_904.wav
/content/data/dataset/dataset/audios_train/audio_661.wav
/content/data/dataset/dataset/audios_train/audio_887.wav
/content/data/dataset/dataset/audios_train/audio_167.wav
/content/data/dataset/dataset/audios_train/audio_808.wav
/content/data/dataset/dataset/audios_train/audio_1082.wav
/content/data/dataset/dataset/audios_train/audio_905.wav
/content/data/dataset/dataset/audios_train/audio_930.wav
/content/data/dataset/dataset/audios_train/audio_886.wav
/content/data/dataset/dataset/audios_train/audio_482.wav
/content/data/dataset/dataset/audios_train/audio_427.wav
/content/data/dataset/dataset/audios_train/audio_278.wav
/content/data/dataset/dataset/audios_train/audio_693.wav
/content/data/dataset/dataset/audios_train/audio_722.wav
/content/dat

In [ ]:
import os
import pandas as pd
import whisper
import language_tool_python

# Set correct path
unzip_dir = "/content/data/dataset/dataset"  # fixed path
TRAIN_AUDIO_DIR = os.path.join(unzip_dir, "audios_train")
TEST_AUDIO_DIR = os.path.join(unzip_dir, "audios_test")

# Load CSVs
train_csv = pd.read_csv(os.path.join(unzip_dir, "train.csv"))
test_csv = pd.read_csv(os.path.join(unzip_dir, "test.csv"))
sample_submission = pd.read_csv(os.path.join(unzip_dir, "sample_submission.csv"))

# Load Whisper and grammar tool
model = whisper.load_model("base", device="cuda")  # or "small" if you want more accuracy
tool = language_tool_python.LanguageTool('en-US')


100%|███████████████████████████████████████| 139M/139M [00:03<00:00, 45.5MiB/s]
INFO:language_tool_python.download_lt:Unzipping /tmp/tmpd7ry3_m5.zip to /root/.cache/language_tool_python.
INFO:language_tool_python.download_lt:Downloaded https://www.languagetool.org/download/LanguageTool-6.5.zip to /root/.cache/language_tool_python.


In [ ]:
train_csv.columns

Index(['filename', 'label'], dtype='object')

In [ ]:
import language_tool_python

tool = language_tool_python.LanguageTool('en-US')


In [ ]:
def get_grammar_score(text):
    matches = tool.check(text)
    num_errors = len(matches)
    num_words = len(text.split())

    if num_words == 0:
        return 1  # Fallback score for empty transcriptions

    errors_per_100_words = (num_errors / num_words) * 100

    # Scoring logic based on error density
    if errors_per_100_words > 30:
        return 1
    elif errors_per_100_words > 20:
        return 2
    elif errors_per_100_words > 10:
        return 3
    elif errors_per_100_words > 5:
        return 4
    else:
        return 5


In [ ]:
!pip install tqdm
from tqdm import tqdm

def process_dataset(df, audio_dir):
    results = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        filename = row['filename']
        audio_path = os.path.join(audio_dir, filename)

        try:
            transcription = model.transcribe(audio_path)["text"]
            score = get_grammar_score(transcription)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            transcription = ""
            score = 1  # default to worst score if error occurs

        results.append({
            "filename": filename,
            "grammar": score
        })

    return pd.DataFrame(results)


In [ ]:
import pandas as pd
import os

unzip_dir = "/content/data/dataset/dataset"
TRAIN_AUDIO_DIR = os.path.join(unzip_dir, "audios_train")

train_csv = pd.read_csv(os.path.join(unzip_dir, "train.csv"))


In [ ]:
import whisper
model = whisper.load_model("base", device="cuda")


In [ ]:
import language_tool_python
tool = language_tool_python.LanguageTool('en-US')

In [ ]:
train_results = process_dataset(train_csv, TRAIN_AUDIO_DIR)


100%|██████████| 444/444 [49:16<00:00,  6.66s/it]


In [ ]:
import pandas as pd
import os

# Path to the extracted dataset
unzip_dir = "/content/data/dataset/dataset"

# Reload CSVs
test_csv = pd.read_csv(os.path.join(unzip_dir, "test.csv"))



In [ ]:
import os

# Set the correct extracted dataset directory
unzip_dir = "/content/data/dataset/dataset"

# Define audio directories
TRAIN_AUDIO_DIR = os.path.join(unzip_dir, "audios_train")
TEST_AUDIO_DIR = os.path.join(unzip_dir, "audios_test")



In [ ]:
test_results = process_dataset(test_csv, TEST_AUDIO_DIR)




100%|██████████| 195/195 [17:05<00:00,  5.26s/it]


In [ ]:
# Save test results to CSV
test_results.to_csv("/content/test_grammar_scores.csv", index=False)

print("Test results saved to test_grammar_scores.csv")


Test results saved to test_grammar_scores.csv
